# 1. Selenium이란?
* 웹브라우저를 자동으로 제어하는 라이브러리
* 원래 다양한 웹브라우저를 자동으로 테스트하는 테스트 도구
* 동적 웹사이트에서 정보를 가져오는데 활용

In [ ]:
# !pip install selenium
# !pip install webdriver-manager

In [1]:
import selenium
print(selenium.__version__)

4.31.1


In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup as bs
import time


# 크롬 옵션즈에 정보를 담아 사람인 것 처럼 만들기
options = webdriver.ChromeOptions()
options.add_experimental_option("excludeSwitches", ['enable-logging'])
options.add_argument("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36")
options.add_argument('lang=ko_KR')

# 크롬 웹브라우저 드라이버 자동 다운로드
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()))
driver.set_window_size(1920, 1080) #웹브라우저 해상도 조절
driver.get("https://search.shopping.naver.com/book/home")

In [2]:
#lnb > div > div._lnbSearch_lnb_search_10s9T > div > div > form > input
# _lnbSearch_search_input_YVyzR N=a:sech.input
serch_box = driver.find_element(By.CSS_SELECTOR, "#lnb > div > div._lnbSearch_lnb_search_10s9T > div > div > form > input")
serch_box.send_keys("핀테크") #+Keys.ENTER)

In [3]:
#lnb > div > div._lnbSearch_lnb_search_10s9T > div > div > form > button
search_button = driver.find_element(By.CSS_SELECTOR, "#lnb > div > div._lnbSearch_lnb_search_10s9T > div > div > form > button")
search_button.click()

In [5]:
result = {}
for page in range(1, 7):
    # 마우스 스크롤로 40개 정보 가져옴
    y = 0
    y_step = 1000
    for scroll_times in range(1, 9):
        y = y + y_step
        driver.execute_script(f"window.scrollTo({0}, {y})")
        time.sleep(2)
    
    page_html = driver.page_source # 40개 정보가 담긴 html 페이지 저장
    soup = bs(page_html, 'lxml') # html 페이지 parsing
    book_list = soup.select(".list_book > li") # 정보 추출
    
    for book in book_list:
        title = book.select_one("span.bookListItem_text__SL9m9").text # 책 제목
        # 저자와 출판사
        if len(book.select("span.bookListItem_define_data__IUMgt")) != 0:
            if book.select_one(".bookListItem_define_item__Jb5MS > span.bookListItem_define_data__IUMgt") != None:  # 저자
                author = book.select_one(".bookListItem_define_item__Jb5MS > span.bookListItem_define_data__IUMgt").text  
            else:
                author = ""
            if book.select_one(".bookListItem_detail_publish__67dDL > span.bookListItem_define_data__IUMgt") != None:  # 출판사
                publisher = book.select_one(".bookListItem_detail_publish__67dDL > span.bookListItem_define_data__IUMgt").text
            else:
                publisher = ""    
        # 출간일
        if book.select_one("div.bookListItem_detail_date__s7KQe") != None:
            pub_date = book.select_one("div.bookListItem_detail_date__s7KQe").text.strip(". ").replace(".","-") # 출간일
        else:
            pub_date = ""
        # 가격부분   
        if book.select("span.bookPrice_price__OagxI > em") != None:
            print("가격부분:",len(book.select("span.bookPrice_price__OagxI > em")))
            if len(book.select("span.bookPrice_price__OagxI > em")) == 2:
                price = book.select("span.bookPrice_price__OagxI > em")[0].text.replace(",","") # 종이책 가격
                eb_price = book.select("span.bookPrice_price__OagxI")[1].text.replace(",","") # e북 가격
            elif len(book.select("span.bookPrice_price__OagxI > em")) == 1:
                price = book.select("span.bookPrice_price__OagxI > em")[0].text.replace(",","") # 종이책 가격
                eb_price = 0
        else:
            price = 0
            eb_price = 0
        
        keys = ('title', 'author', 'publisher', 'pub_date', 'price', 'eb_price')
        data = (title, author, publisher, pub_date, price, eb_price)
        for key, item in zip(keys, data):
            result.setdefault(key, []).append(item)
    
    # 다음 페이지 이동 버튼 누르기
    npbutton = "div.Paginator_list_paging__hUKRc > button.Paginator_btn_next__7NiBL"
    next_page_button = driver.find_element(By.CSS_SELECTOR, npbutton)
    next_page_button.click()
driver.close()

가격부분: 2
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 0
가격부분: 0
가격부분: 0
가격부분: 0
가격부분: 0
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 0
가격부분: 1
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 2
가격부분: 2
가격부분: 0
가격부분: 1
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 0
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 0
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 0
가격부분: 2
가격부분: 0
가격부분: 0
가격부분: 1
가격부분: 1
가격부분: 0
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 0
가격부분: 2
가격부분: 2
가격부분: 0
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 0
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 2
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 2
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1
가격부분: 1


In [6]:
import pandas as pd
book_result = pd.DataFrame(result)
book_result

,title,author,publisher,pub_date,price,eb_price
0,핀테크,한석주,커뮤니케이션북스,2015-11-01,11400,구매 8640원
1,핀테크 (IT와 금융이 만나는 새로운 세상),강창호,한빛미디어,2015-06-05,12600,0
2,핀테크 전쟁 (세계 금융 시장의 거대한 트렌드 | 새로운 돈의 시대가 온다),브렛 킹,예문,2015-06-12,13500,구매 9450원
3,핀테크 4.0 (핀테크 혁명과 금융의 미래),김종현,한국금융연수원,2022-05-24,17100,0
4,핀테크 혁명 (모바일 은행 설립자가 알려주는 핀테크 시대의 돈 관리 기술),앤 보덴,유엑스리뷰,2020-03-03,20700,0
...,...,...,...,...,...,...
235,"핀테크 트렌드 2024 : IT·금융권 취업을 위한, 핀테크 현황과 전망 총망라!",좋은정보사,좋은정보사,,17100,0
236,"핀테크 트렌드 2024 : IT·금융권 취업을 위한, 핀테크 현황과 전망 총망라!",좋은정보사,좋은정보사,,17100,0
237,JOB 스페셜 15~17권(전3권/신소재+핀테크+주식투자)미래탐험꿈발전소,,국일아이(전집),2021-02-28,34560,0
238,"[보리보리/밀크북]부자, 관상, 기술 부자들은 알고 있는 핀테크 시대의 행동경제학",,국일아이(전집),,14400,0


# 마우스를 스크롤 해서 40개 정보 가져오기
* window.scrollTo({시작위치}, {step})

In [13]:
#container > div > div.bookSearch_book_search__Fm5Em > div > div.Paginator_list_paging__hUKRc > button
#container > div > div.bookSearch_book_search__Fm5Em > div > div.Paginator_list_paging__hUKRc > button
next_page_button = driver.find_element(By.CSS_SELECTOR, "div > div.Paginator_list_paging__hUKRc > button")
next_page_button.click()

In [23]:
book.select("span.bookPrice_price__OagxI")

[<span class="bookPrice_price__OagxI"><em>2,790</em>원<span class="bookPrice_delivery__odeYG"><svg class="bookPrice_svg_delivery__3BcGQ" height="11" width="14" xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink"><defs><path d="M0 0h14v11H0z" id="IconBookListDelivery_svg__a"></path></defs><g fill="none" fill-rule="evenodd" transform="matrix(-1 0 0 1 14 0)"><mask fill="#fff" id="IconBookListDelivery_svg__b"><use xlink:href="#IconBookListDelivery_svg__a"></use></mask><path d="M13.296 8.779c0 .02-.02.04-.04.04h-.685c-.163-.82-.85-1.436-1.676-1.437C10.07 7.383 9.383 8 9.22 8.82H4.78C4.618 8 3.93 7.383 3.105 7.383 2.28 7.383 1.592 8 1.43 8.82H.743c-.019 0-.039-.021-.039-.041v-2.91a1.97 1.97 0 0 1 .133-.626l1.149-2.55c.006-.052.194-.177.24-.16h3.28c.02 0 .04.02.04.042v4.078h7.75V8.78zm-2.4 1.477c-.557 0-1.007-.476-1.009-1.064.002-.589.452-1.064 1.008-1.065.558 0 1.008.476 1.01 1.065-.002.588-.452 1.063-1.01 1.064zm-7.792 0c-.557-.001-1.008-.476-1.008-1.064 0-.589.45-1